---
title: Hands on Dulwich
created: '2020-11-29T21:13:47-08:00'
date: '2026-07-19T10:55:42-07:00'
authors:
  - bendu
label: hands-on-dulwich
license: CC-BY-4.0
tags:
  - computer science
  - programming
  - Python
  - Git
  - Dulwicch
  - version control
---

**Things on this page are fragmentary and immature notes/thoughts of the author. Please read with your own judgement!**


Note: dulwich is not feature complete yet 
and the development of the project is extremely slow.
It is suggested that you use other Python packages instead.
For more discussions,
please refer to
[Git Implementations and Bindings in Python](git-implementations-and-bindings-in-python)
.

## Tips and Traps

1. The `git` command (and thus Dulwich) accepts URLs both with and without the trailing `.git`. 

In [1]:
!pip3 install dulwich

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.2 -> 23.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [1]:
from pathlib import Path
from dulwich import porcelain
from dulwich.repo import Repo
from dulwich.walk import WalkEntry

In [51]:
url = "https://github.com/dclong/test_dulwich"
dir_local = Path("/tmp/test_dulwich")
!rm -rf {dir_local}

## git clone

In [52]:
repo = porcelain.clone(url, dir_local)

In [53]:
repo

<Repo at '/tmp/test_dulwich'>

In [54]:
!ls {dir_local}

abc  build.sh  readme.md  test1.txt


## git fetch

In [38]:
porcelain.fetch(repo=dir_local)

FetchPackResult({b'HEAD': b'729bb376c018f068548c574a9fa05764432ef33a', b'refs/heads/dev': b'9f02363cbb049bdc6d0384c78d9b581dc2e31dba', b'refs/heads/feature': b'eab89ba3900c7483d97978f5038cea68997f6780', b'refs/heads/main': b'729bb376c018f068548c574a9fa05764432ef33a', b'refs/pull/1/head': b'9f02363cbb049bdc6d0384c78d9b581dc2e31dba', b'refs/tags/v1.0.0': b'6716bb0d016bd63ba543f3d9c67a65dadecd152e', b'refs/tags/v1.1.0': b'729bb376c018f068548c574a9fa05764432ef33a'}, {b'HEAD': b'refs/heads/main'}, b'git/github-60d715541676-Linux\n')

## dulwich.repo.Repo

In [20]:
type(repo)

dulwich.repo.Repo

Create a repo from a local directory.

In [55]:
repo.path

'/tmp/test_dulwich'

In [8]:
repo2 = Repo("/tmp/test_dulwich")
repo2

<Repo at '/tmp/test_dulwich'>

In [10]:
repo3 = Repo("/workdir/archives/github_actions_scripts")
repo3

<Repo at '/workdir/archives/github_actions_scripts'>

## git status

In [20]:
s = porcelain.status(repo3)

In [21]:
[
    m for m in dir(s) if not m.startswith("_")
]

['count', 'index', 'staged', 'unstaged', 'untracked']

In [22]:
s.staged

{'add': [b'update_version_dockerfile.py'], 'delete': [], 'modify': []}

In [23]:
s.unstaged

[b'update_version_dockerfile.py']

In [49]:
s.untracked

[]

In [76]:
!git -C {dir_local} status

On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   test1.txt

no changes added to commit (use "git add" and/or "git commit -a")


In [26]:
porcelain.get_tree_changes(repo3)

{'add': [b'update_version_dockerfile.py'], 'delete': [], 'modify': []}

In [56]:
!git -C {dir_local} diff

diff --git a/test1.txt b/test1.txt
index 9daeafb..a5f96b1 100644
--- a/test1.txt
+++ b/test1.txt
@@ -1 +1,3 @@
 test
+add a new line
+


In [64]:
repo[b"head"]

KeyError: b'head'

In [66]:
repo.head

<bound method BaseRepo.head of <Repo at '/tmp/test_dulwich'>>

In [68]:
repo.head()

b'729bb376c018f068548c574a9fa05764432ef33a'

In [72]:
repo[repo.head()]

<Commit b'729bb376c018f068548c574a9fa05764432ef33a'>

In [73]:
repo.object_store

<DiskObjectStore('/tmp/test_dulwich/.git/objects')>

In [74]:
import sys

from dulwich.patch import write_tree_diff

outstream = getattr(sys.stdout, 'buffer', sys.stdout)
write_tree_diff(outstream, repo.object_store, repo[repo.head()], commit.tree)

NameError: name 'commit' is not defined

In [ ]:
import sys

from dulwich.patch import write_tree_diff
from dulwich.repo import Repo

repo_path = "."
commit_id = b"a6602654997420bcfd0bee2a0563d9416afe34b4"

r = Repo(repo_path)

commit = r[commit_id]
parent_commit = r[commit.parents[0]]
outstream = getattr(sys.stdout, 'buffer', sys.stdout)
write_tree_diff(outstream, r.object_store, parent_commit.tree, commit.tree)

## git add

In [33]:
!touch /tmp/test_dulwich/abc2

In [34]:
!ls /tmp/test_dulwich/

abc  abc2  build.sh  Dockerfile  LICENSE  readme.md  scripts


In [35]:
!git -C /tmp/test_dulwich/ status

On branch dev
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   abc

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	abc2



### [dulwich.porcelain.add](https://www.dulwich.io/api/dulwich.porcelain.html#add)

This is the recommended way to add changes to stage.
`Repo.stage` and `Repo.unstage` have been deprecated and will be removed in v0.26.0+.
Notice that the original behavior of `porcelain.add` 
was to add files in the current working directory of Python 
instead of files from the specified repository (if different from the current working directory),
which was confusing.
This was later fixed.
For more details,
please refer to the 
[issue](https://github.com/dulwich/dulwich/issues/895) 
that I filed.

In [32]:
porcelain.add("/tmp/test_dulwich", paths="/tmp/test_dulwich/abc")

(['abc'], set())

In [30]:
porcelain.add(repo3, paths="update_version_dockerfile.py")

(['update_version_dockerfile.py'], set())

In [33]:
porcelain.add(repo3)

(['nima'], set())

By default,
dulwich adds all files in the current working directory,
which is not the right behavior!
I have submitted [a ticket](https://github.com/dulwich/dulwich/issues/895) to fix the issue.

In [67]:
porcelain.add("/tmp/test_dulwich", paths="/tmp/test_dulwich/.")

(['./'], set())

In [71]:
status = porcelain.status("/tmp/test_dulwich")
status

GitStatus(staged={'add': [], 'delete': [], 'modify': []}, unstaged=[], untracked=['abc2'])

In [72]:
status.unstaged

[]

In [73]:
status.untracked

['abc2']

## git commit

### [dulwich.porcelain.commit](https://www.dulwich.io/api/dulwich.porcelain.html#commit)

In [34]:
porcelain.commit(repo3, message="update python script")

b'b616e430f08642973571c51c9c5eeced0f0f17eb'

In [50]:
porcelain.commit("/tmp/test_dulwich/", message="add abc")

b'11fb9f18f9d211c93175e898faa731584b8be368'

## ConfigFile

ConfigFile inherits ConfigDict which means that you operate on a ConfiFile like a `dict`.

In [14]:
config = repo.get_config()
config

ConfigFile(CaseInsensitiveDict([((b'core',), CaseInsensitiveDict([(b'repositoryformatversion', b'0'), (b'filemode', b'true'), (b'bare', b'false'), (b'logallrefupdates', b'true')])), ((b'remote', b'origin'), CaseInsensitiveDict([(b'url', b'https://github.com/dclong/docker-ubuntu_b'), (b'fetch', b'+refs/heads/*:refs/remotes/origin/*')])), ((b'remote', b'nima'), CaseInsensitiveDict([(b'url', b'https://github.com/dclong/docker-ubuntu_b'), (b'fetch', b'+refs/heads/*:refs/remotes/nima/*')]))]))

In [37]:
config.keys()

KeysView(ConfigFile(CaseInsensitiveDict([((b'core',), CaseInsensitiveDict([(b'repositoryformatversion', b'0'), (b'filemode', b'true'), (b'bare', b'false'), (b'logallrefupdates', b'true')])), ((b'remote', b'origin'), CaseInsensitiveDict([(b'url', b'https://github.com/dclong/docker-ubuntu_b'), (b'fetch', b'+refs/heads/*:refs/remotes/origin/*')])), ((b'remote', b'nima'), CaseInsensitiveDict([(b'url', b'https://github.com/dclong/docker-ubuntu_b'), (b'fetch', b'+refs/heads/*:refs/remotes/nima/*')]))])))

In [38]:
config.values()

ValuesView(ConfigFile(CaseInsensitiveDict([((b'core',), CaseInsensitiveDict([(b'repositoryformatversion', b'0'), (b'filemode', b'true'), (b'bare', b'false'), (b'logallrefupdates', b'true')])), ((b'remote', b'origin'), CaseInsensitiveDict([(b'url', b'https://github.com/dclong/docker-ubuntu_b'), (b'fetch', b'+refs/heads/*:refs/remotes/origin/*')])), ((b'remote', b'nima'), CaseInsensitiveDict([(b'url', b'https://github.com/dclong/docker-ubuntu_b'), (b'fetch', b'+refs/heads/*:refs/remotes/nima/*')]))])))

In [40]:
config[(b"remote", b"origin")]

CaseInsensitiveDict([(b'url', b'https://github.com/dclong/docker-ubuntu_b'),
                     (b'fetch', b'+refs/heads/*:refs/remotes/origin/*')])

In [27]:
config.get((b"remote", b"origin"), b"url")

b'https://github.com/dclong/docker-ubuntu_b'

In [31]:
dict(config)

{(b'core',): CaseInsensitiveDict([(b'repositoryformatversion', b'0'),
                      (b'filemode', b'true'),
                      (b'bare', b'false'),
                      (b'logallrefupdates', b'true')]),
 (b'remote',
  b'origin'): CaseInsensitiveDict([(b'url',
                       b'https://github.com/dclong/docker-ubuntu_b'),
                      (b'fetch', b'+refs/heads/*:refs/remotes/origin/*')]),
 (b'remote',
  b'nima'): CaseInsensitiveDict([(b'url',
                       b'https://github.com/dclong/docker-ubuntu_b'),
                      (b'fetch', b'+refs/heads/*:refs/remotes/nima/*')])}

## git remote -v

In [31]:
[m for m in dir(dulwich.porcelain) if 'remote' in m]

['_import_remote_refs',
 'get_branch_remote',
 'get_remote_repo',
 'ls_remote',
 'remote_add',
 'remote_remove']

In [34]:
porcelain.get_branch_remote(repo)

b'origin'

In [36]:
porcelain.get_remote_repo(repo)

('origin', 'https://github.com/dclong/docker-ubuntu_b')

In [39]:
porcelain.ls_remote(url)

{b'HEAD': b'4996e93a5f24c375b1d56deddda1cd9cfddd14f6',
 b'refs/heads/centos7': b'd7a6f672771be1fc33ddd1006b3318901c914b19',
 b'refs/heads/debian': b'c7fc15b52f4c76faa4ba487a113b5c987ac3b371',
 b'refs/heads/dev': b'4996e93a5f24c375b1d56deddda1cd9cfddd14f6',
 b'refs/heads/main': b'925dd68d39ea943f1c387e4906e72aebc765c4bf',
 b'refs/pull/1/head': b'b90bd029b4707a94640957cbfae73a631a9d83e0',
 b'refs/pull/1/merge': b'c767a6a929e599c245a4241477093554c1ab0d10',
 b'refs/pull/10/head': b'ea5b60d7fc34fce65ec4a503c20536d3e0d4587a',
 b'refs/pull/100/head': b'b514f4d74dd93b237cb72a91e992a0480393b546',
 b'refs/pull/101/head': b'b514f4d74dd93b237cb72a91e992a0480393b546',
 b'refs/pull/102/head': b'f2f28a5809657930caa51d2606dc62c3e74d2e27',
 b'refs/pull/103/head': b'f2f28a5809657930caa51d2606dc62c3e74d2e27',
 b'refs/pull/104/head': b'43863e528cbd069b8f095f8eade79ae990fbe916',
 b'refs/pull/105/head': b'43863e528cbd069b8f095f8eade79ae990fbe916',
 b'refs/pull/106/head': b'238b7db8c24f6ed43125696405f56cbd33

In [36]:
[key[1].decode() for key in config.keys() if key[0] == b"remote"]

['origin', 'nima']

## dulwich.porcelain.ls_remote

In [79]:
porcelain.ls_remote(url)

{b'HEAD': b'2fd55f0a653bf8ec2e7ffda16c7b2d601167da06',
 b'refs/heads/debian': b'618389f8300615ba7d31c4ba7b75fb94770391e1',
 b'refs/heads/dev': b'2fd55f0a653bf8ec2e7ffda16c7b2d601167da06',
 b'refs/heads/main': b'71f2b1d96cfa8596319686a5a98514ad4ac85506',
 b'refs/pull/1/head': b'b90bd029b4707a94640957cbfae73a631a9d83e0',
 b'refs/pull/1/merge': b'c767a6a929e599c245a4241477093554c1ab0d10',
 b'refs/pull/10/head': b'ea5b60d7fc34fce65ec4a503c20536d3e0d4587a',
 b'refs/pull/11/head': b'0d7e63476077d2ed51728823f3ce54b57a8287e6',
 b'refs/pull/12/head': b'd2fcee067c4f003084950799f8cb408625d8c610',
 b'refs/pull/13/head': b'5d29d66bdadfb7e265b0f895ca1bf6f26f7bad39',
 b'refs/pull/14/head': b'114a4a2af63270ea69f08b5801cf1bdfa1c29222',
 b'refs/pull/15/head': b'ffe1f38e3a91ef69b784b867ad5d7729bf17499f',
 b'refs/pull/16/head': b'1b2ee6dbbe96435009ef96a80776ebc30e74b082',
 b'refs/pull/17/head': b'b1d52163d2170c1b7a8fe9a0d05bf15a7c4dfcb0',
 b'refs/pull/18/head': b'ac8a2c93c8c93985eb7a6e2742e2a3e5aa4786d8',

## dulwich.objects.Commit 

In [43]:
repo.head()

b'2fd55f0a653bf8ec2e7ffda16c7b2d601167da06'

In [45]:
commit = repo[repo.head()]
commit

<Commit b'2fd55f0a653bf8ec2e7ffda16c7b2d601167da06'>

In [46]:
type(commit)

dulwich.objects.Commit

In [39]:
commit.message

b'Update etc.sh'

In [41]:
main = heads.main
main

NameError: name 'heads' is not defined

Get the commit pointed to by head called master.

In [17]:
main.commit

<git.Commit "95ed236bd715a06320ee85d519fb79a0adffe072">

In [18]:
main.rename("main2")

<git.Head "refs/heads/main2">

Verify that the `main` branch has been renamed to `main2`.

In [19]:
!cd {dir_local} && git branch

* main2


## [dulwich.porcelain.active_branch](https://www.dulwich.io/api/dulwich.porcelain.html#active_branch)

In [20]:
porcelain.active_branch(repo)

b'main'

On a respoistory initiated with `jj git init --colocate`.

In [9]:
porcelain.active_branch(Path.home() / "tmp")

b'main'

In deatched HEAD mode (e.g., in a jj + git respository after jj creating commits),

In [11]:
porcelain.active_branch(Path.home() / "tmp")

On a respoistory initiated with `git init`.

In [10]:
porcelain.active_branch(Path.home() / "tmp2")

b'master'

## [dulwich.porcelain.branch_list](https://www.dulwich.io/api/dulwich.porcelain.html#branch_list)

In [21]:
porcelain.branch_list(repo)

{b'main'}

## [dulwich.porcelain.branch_create](https://www.dulwich.io/api/dulwich.porcelain.html#branch_create)

Create a new branch.
Notice that this method won't automatically switch to the newly created branch.
You have to do it manually using 
[dulwich.porcelain.checkout](https://www.dulwich.io/api/dulwich.porcelain.html#checkout)
.

In [ ]:
porcelain.branch_create(repo=".", name="new_branch")

## [dulwich.porcelain.checkout](https://www.dulwich.io/api/dulwich.porcelain.html#checkout)

Switch to a branch or commit, updating both HEAD and the working tree.

In [ ]:
porcelain.checkout(repo=".", target="new_branch")

## Changed Files

Update a file.

In [23]:
!echo "# add a line of comment" >> {dir_local}/build.sh

## Staged Files

The file `build.sh` is now staged.

Commit the change.

## git push

### [dulwich.porcelain.push](https://www.dulwich.io/api/dulwich.porcelain.html#push)

In [36]:
porcelain.push(repo3)

SendPackResult({b'refs/heads/main': b'b616e430f08642973571c51c9c5eeced0f0f17eb'}, b'github/spokes-receive-pack-acac8763c60f636c44baaf5c3887895cf5f55c30')

## git pull

In [11]:
!ls {dir_local}

abc  build.sh  readme.md


In [12]:
porcelain.pull(repo, refspecs="main")

In [13]:
!ls {dir_local}

abc  build.sh  readme.md  test1.txt


## git checkout

In [12]:
[m for m in dir(porcelain) if 'checkout' in m]

['_update_head_during_checkout_branch', 'checkout_branch']

In [18]:
!git -C {dir_local} status

On branch dev
nothing to commit, working tree clean


In [26]:
!git -C {dir_local} branch

* dev
  main


In [27]:
porcelain.checkout_branch(repo, "main")

In [28]:
!git -C {dir_local} branch

  dev
* main


## git tag
List all tags.

In [27]:
!git -C {dir_local} tag 

v1.0.0


In [24]:
[m for m in dir(porcelain) if "tag" in m]

['_make_tag_ref',
 'get_unstaged_changes',
 'print_tag',
 'show_tag',
 'tag_create',
 'tag_delete',
 'tag_list']

In [37]:
porcelain.tag_list(repo)

[b'v1.0.0', b'v2.0.0']

## git tag tag_name

Create a new tag.

In [38]:
porcelain.tag_create(repo, "v2.0.0")

In [40]:
porcelain.push(repo, refspecs="v2.0.0")

HTTPUnauthorized: No valid credentials provided

## git diff

In [16]:
help(repo.refs[4].commit.diff)

Help on method diff in module git.diff:

diff(other: Union[Type[git.diff.Diffable.Index], Type[ForwardRef('Tree')], object, NoneType, str] = <class 'git.diff.Diffable.Index'>, paths: Union[str, List[str], Tuple[str, ...], NoneType] = None, create_patch: bool = False, **kwargs: Any) -> 'DiffIndex' method of git.objects.commit.Commit instance
    Creates diffs between two items being trees, trees and index or an
    index and the working tree. It will detect renames automatically.
    
    :param other:
        Is the item to compare us with.
        If None, we will be compared to the working tree.
        If Treeish, it will be compared against the respective tree
        If Index ( type ), it will be compared against the index.
        If git.NULL_TREE, it will compare against the empty tree.
        It defaults to Index to assure the method will not by-default fail
        on bare repositories.
    
    :param paths:
        is a list of paths or a single path to limit the diff to.
 

In [3]:
url = "https://github.com/dclong/docker-ubuntu_b.git"
dir_local = "/tmp/" + url[(url.rindex("/") + 1) :]
!rm -rf {dir_local}

In [4]:
repo = git.Repo.clone_from(url, dir_local, branch="main")
repo

<git.repo.base.Repo '/tmp/docker-ubuntu_b.git/.git'>

In [25]:
repo.refs

[<git.Head "refs/heads/debian">,
 <git.Head "refs/heads/dev">,
 <git.Head "refs/heads/main">,
 <git.RemoteReference "refs/remotes/origin/HEAD">,
 <git.RemoteReference "refs/remotes/origin/debian">,
 <git.RemoteReference "refs/remotes/origin/dev">,
 <git.RemoteReference "refs/remotes/origin/main">]

In [6]:
diffs = repo.refs[4].commit.diff(repo.refs[3].commit)
diffs

[]

In [21]:
diffs = repo.refs[4].commit.diff(repo.refs[2].commit)
diffs

[<git.diff.Diff at 0x7f0eb05d1a60>, <git.diff.Diff at 0x7f0eb05d1af0>]

In [13]:
str(diffs[0])

'Dockerfile\n=======================================================\nlhs: 100644 | 8ae5c7650a8c031a8e176d896a3665bbe7e2aae8\nrhs: 100644 | 9f2304d9a97aa1279ad1938b3bb74790172c9d8b'

In [12]:
repo.refs[5].name

'origin/main'

In [6]:
print(repo.git.status())

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [6]:
repo.git.checkout("debian", force=True)

"Your branch is up to date with 'origin/debian'."

In [8]:
repo.git.checkout(b="a_new_branch", force=True)

''

In [ ]:
nima = repo.refs[4].checkout(force=True, b="nima")
nima

In [50]:
diffs = nima.commit.diff(repo.refs[-1].commit)
diffs[0].diff

''

Diff the `dev` and the `main` branch,
which is equivalent to the Git command 
`git diff dev..main`.

In [30]:
repo.refs[2].commit.diff(repo.refs[1].commit)

[]

In [32]:
diffs = repo.refs[2].commit.diff(repo.refs[0].commit)
diffs

In [33]:
diffs[0]

In [24]:
diffs = repo.refs[6].commit.diff(repo.refs[7].commit)
diffs

[]

In [25]:
diffs = repo.refs[4].commit.diff(repo.refs[7].commit)
diffs

In [26]:
diffs[0].diff

''

In [27]:
diffs = repo.refs[7].commit.diff(repo.refs[4].commit)
diffs

In [28]:
diffs[0].diff

''

In [19]:
any(ele for ele in [""])

False

In [23]:
repo.branches[0].name

'dev'

In [ ]:
for branch in repo.branches:
    branch.

In [12]:
commit = repo.head.commit
commit

<git.Commit "6716bb0d016bd63ba543f3d9c67a65dadecd152e">

In [15]:
type(repo.branches[0])

git.refs.head.Head

In [17]:
repo.refs[4].commit.diff(repo.refs[2].commit)

[]

In [9]:
repo.refs[4].commit.diff(repo.refs[3].commit)

In [20]:
help(repo.git.branch)

Help on function <lambda> in module git.cmd:

<lambda> lambda *args, **kwargs



In [28]:
repo.heads

[<git.Head "refs/heads/dev">, <git.Head "refs/heads/main">]

Diff the `debian` and the `main` branches but limit diff to specified paths 
(via the `paths` parameter).

In [24]:
diffs = repo.refs[4].commit.diff(repo.refs[2].commit, paths=["build.sh", "scripts"])
diffs

[]

## Get Ahead Commits

In [2]:
def get_remote_refs(repo, remote: str = "origin") -> dict[bytes, bytes]:
    url = repo.get_config().get(("remote", remote), "url").decode("utf-8")
    return porcelain.ls_remote(url).refs


def get_ahead_commits(path: str, branch: str = "", remote: str = "origin") -> list[WalkEntry]:
    repo = Repo(path)
    if not branch:
        branch = porcelain.active_branch(repo)
    if isinstance(branch, str):
        branch = branch.encode()
    remote_branch_head = get_remote_refs(repo=repo, remote=remote)[b"refs/heads/" + branch]
    walker = repo.get_walker(
        include=[repo.head()],
        exclude=[remote_branch_head]
    )
    return list(walker)

In [3]:
commits = get_ahead_commits("test_dulwich")
commits

[<WalkEntry commit=1e000c9b6b2557d54f08623ca470241352b6fc2a, changes=[TreeChange(type='modify', old=TreeEntry(path=b'abc', mode=33188, sha=b'ed1293c7cab82424dcee8060829d7345a49e6108'), new=TreeEntry(path=b'abc', mode=33188, sha=b'a88d44100e656261166feab9f690f3b2f4b4d210'))]>]

## References

https://github.com/dulwich/dulwich

https://www.dulwich.io/docs/api/